<a href="https://colab.research.google.com/github/lilydeller/Math344-Project/blob/main/notebooks/03_low_rank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part A

Clone our project repository and import functions from Milestone 2.

In [ ]:
import subprocess
import sys
from pathlib import Path

USER = "lilydeller"
REPO = "Math344-Project"

if Path(REPO).exists():
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(
        ["git", "clone", f"https://github.com/{USER}/{REPO}.git", REPO], check=True
    )

sys.path.insert(0, f"{REPO}/src")

from evaluate import split_ratings, to_matrix, rmse
from models import bias_baseline

Download the movie dataset and create our training data, testing data, fitting data, and validation data.

In [ ]:
from io import BytesIO
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

ML_DATASET = Path("ml-100k")

if not ML_DATASET.exists():
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    response = requests.get(url)

    ZipFile(BytesIO(response.content)).extractall()

movie_ratings = pd.read_csv(
    ML_DATASET / "u.data", sep="\t", names=["user", "item", "rating", "time"]
)

train, test = split_ratings(movie_ratings, seed=0)
fit, val = split_ratings(train, seed=1)
print("fit: ", len(fit), " validation: ", len(val), " test: ", len(test))

fit:  81200  validation:  8832  test:  9968


# Part B

In [ ]:
R_fit = to_matrix(fit)
known_fit = ~np.isnan(R_fit)
val_truth = val["rating"].to_numpy()
val_u = val["user"].to_numpy() - 1
val_i = val["item"].to_numpy() - 1

def score_at(P, rows, cols, truth):
    """RMSE for selected entries of a prediction matrix."""
    pred = np.clip(P[rows, cols], 1, 5)
    return rmse(pred, truth)

# Part C

In [ ]:
def baseline_matrix(R):
    """Full matrix: overall mean plus user and movie effects."""
    mu, b_user, b_item = bias_baseline(R)
    return mu + b_user[:, None] + b_item[None, :]

B_fit = baseline_matrix(R_fit)
baseline_val_rmse = score_at(B_fit, val_u, val_i, val_truth)
print("Baseline validation RMSE: ", round(baseline_val_rmse, 4))

# Part D

In [ ]:
def rank_k(U, s, Vt, k):
    """Build a rank-k matrix from an already computed SVD."""
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

#Part E: Model 1

Fills the missing entries in the fitting data with the movie's average rating so an SVD can be computed on it.

In [ ]:
mu_fit = np.nanmean(R_fit)
n_item = known_fit.sum(axis=0)
item_mean = np.full(R_fit.shape[1], mu_fit)
has_rating = n_item > 0
item_mean[has_rating] = (
np.nansum(R_fit, axis=0)[has_rating] / n_item[has_rating])
F = np.where(known_fit, R_fit, item_mean[None, :])
U1, s1, Vt1 = np.linalg.svd(F, full_matrices=False)

#Part F: Model 2

Gets the rating's difference between the actual (R fit) and the baseline ratings (B fit), marking a difference of 0 for where a user didn't leave a rating. Then, using an SVD, forms a low-rank approx on these differences.

In [ ]:
X_fit = np.where(known_fit, R_fit - B_fit, 0.0)
U2, s2, Vt2 = np.linalg.svd(X_fit, full_matrices=False)